# Clustering Optimization (Exercise)

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/M0/01-clustering/clustering_optimization_exercise.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>


In [ ]:
# --- Setup: Clone repo & cd into correct folder (for lab setup convenience) ---
import os
import subprocess

repo_url = "https://github.com/HassanAlgoz/dl.git"
lab_folder = "dl/modules/M0/01-clustering"

# Only clone if the folder doesn't exist
if not os.path.exists(lab_folder):
    subprocess.run(["git", "clone", repo_url])

# Change working directory to the lab folder
os.chdir(lab_folder)


## Context

[**Old Faithful** Dataset](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/faithful.html): Waiting times between eruptions and the duration of each eruption for the Old Faithful geyser in Yellowstone National Park, Wyoming, USA. The version used here contains 272 observations with two real-valued attributes.

- Old Faithful is a cone geyser in Yellowstone's Upper Geyser Basin and one of the park's most predictable thermal features, erupting at fairly regular intervals.

- Each row records one complete eruption: how long it lasted (`duration`, in minutes) and how long until the next one (`waiting`, in minutes).

- The data show two distinct eruption regimes — shorter eruptions tend to be followed by shorter waits, while longer eruptions tend to be followed by longer waits — which makes this a classic dataset for visualizing and validating clustering methods.

- This version matches the one bundled with R's `faithful` dataset (column `eruptions` renamed to `duration`), widely used in statistics and machine learning courses.

> Source: Azzalini, A. and Bowman, A. W. (1990). *A look at some data on the Old Faithful geyser.* Applied Statistics, 39, 357–365.

# Imports

In [ ]:
# %pip install -qqq numpy pandas scikit-learn matplotlib seaborn


In [ ]:
# Import what you need from sklearn, plus numpy, pandas, matplotlib, and seaborn.
# Hint: you'll need KMeans, silhouette_score, and StandardScaler

from sklearn.cluster import ...
from sklearn.metrics import ...
from sklearn.preprocessing import ...

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib

matplotlib.style.use('ggplot')


# Load Data

Load the data and take a first look. Do you think this data could benefit from clustering?

In [ ]:
df = pd.read_csv("../data/geyser.csv")
# Take a look at the first few rows


In [ ]:
# Plot duration vs. waiting as a scatter plot
# Hint: df.plot(kind="scatter", ...) or plt.scatter(...)



In [ ]:
# Define X — select the feature columns for clustering
X = ...


In [ ]:
# There's a VERY important step before fitting our K-Means Model. Any ideas?
# Hint: scale your features so each has mean 0 and standard deviation 1

sc = ...
X_scaled = ...


In [ ]:
# Fit a K-means clustering model
# Hint: how many groups do you see in the scatter plot?

km = KMeans(...)
km.fit(...)


## What did we discover?

### I. Cluster Predictions

In [ ]:
# Option 1: Class attribute

# km.???


In [ ]:
# Option 2: Predict

# km.???(...)


In [ ]:
# Attach predicted cluster to original points

df['cluster'] = ...
df.head()


### II. Centroids

In [ ]:
# Option 1: Using groupby

df.groupby('cluster').mean()


In [ ]:
# Option 2: (Preferred method) Using cluster_centers_

km.cluster_centers_

# But wait... why does this not match the one above?!


In [ ]:
# Remember, we scaled this! We need to "unscale".
# Hint: use sc.inverse_transform(...)

centroids = ...


## Visually Verifying Cluster Labels

In [ ]:
# Turn centroids into a DataFrame with columns "duration" and "waiting"

centroids = pd.DataFrame(
    ...,
    columns=["duration", "waiting"]
)
centroids


In [ ]:
plt.figure(figsize=(10, 8))

# Map each cluster to a color (you have 2 clusters)
colors = ["red", "blue"]
df["color"] = df["cluster"].map(lambda p: colors[p])

ax = df.plot(
    kind="scatter",
    x="duration", y="waiting",
    figsize=(10, 8),
    c=df["color"]
)

# Plot centroids on top with marker="*"
centroids.plot(
    kind="scatter",
    x="duration", y="waiting",
    marker="*", c=..., s=550, ax=ax
);


# Metrics for Assessing Clusters
---
## Inertia

Sum of squared differences between each point in a cluster and that cluster's centroid.

How dense is each cluster? 


- low inertia = dense cluster
- ranges from 0 to very high values


$$ \sum_{j=0}^{n} (x_j - \mu_i)^2 $$

where $\mu_i$ is a cluster centroid



`.inertia_` is an attribute of a fitted sklearn's kmeans object

#### Lower inertia is better! Though be cautious with adding too many clusters.

#### Get the inertia

In [ ]:
# Access the inertia of your fitted model
# km.???


---
## Silhouette Score

Tells you how much closer data points are to their own clusters than to the nearest neighbor cluster.

How far apart are the clusters?
- ranges from -1 to 1
- high silhouette score means the clusters are well separated



### $s_i = \frac{b_i - a_i}{max\{a_i, b_i\}}$

Where:
- $a_i$ = Cohesion: Mean distance of points within a cluster from each other.
- $b_i$ = Separation: Mean distance from point $x_i$ to all points in the next nearest cluster.

Use scikit-learn: `metrics.silhouette_score(X_scaled, labels)`.

#### Higher silhouette score is better!


#### Compute the silhouette score 

In [ ]:
# Already imported above, but you can import again if needed
from sklearn.metrics import silhouette_score


In [ ]:
silhouette_score(...)


# Using the elbow method to choose _k_
---

The [elbow method](https://en.wikipedia.org/wiki/Elbow_method_(clustering)) is one possible method to help narrow in on the ideal value of **K**. 

### Choose the number of clusters where the next cluster doesn't significantly improve performance. 

#### Let's use the `.inertia_` method as our evaluation metric

In [ ]:
inertia_list = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    # fit the model and append its inertia to inertia_list
    ...

inertia_list


#### Analyze the above

- Do you see the **"elbow"**?

- What's the best value for _k_?




In [ ]:
plt.plot(range(1, 11), inertia_list, marker='o')
plt.xlabel('# of Clusters')
plt.ylabel('Score')
plt.title('Inertia Scores');


#### Now let's make a plot with the silhouette score

## Remember: Higher is Better for Silhouette Score!

In [ ]:
silhouette_list = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    # fit the model and append the silhouette score to silhouette_list
    ...

silhouette_list


In [ ]:
plt.plot(range(2, 11), silhouette_list, marker='o')
plt.xlabel('# of Clusters')
plt.ylabel('Score')
plt.title('Silhouette Scores');


#### What _k_ should we choose?

_Your answer here._